In [2]:
#!/usr/bin/env python3

import os
import json
import time
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, mean_squared_error

from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.svm import OneClassSVM

from sklearn.linear_model import LogisticRegression, LinearRegression

from sklearn.neural_network import MLPClassifier, MLPRegressor

from xgboost import XGBClassifier, XGBRegressor


########################################
# CONFIG
########################################

DATA_PATH="data"

DATASETS=[
"Bank_Customer_Churn_Dataset",
"cmc",
"connect-4",
"credit-g",
"electricity",
"elevators",
"eucalyptus",
"eye_movements",
"house_16H",
"kc1",
"phoneme",
"pol",
"socmob",
"splice",
"vehicle"
]

MODELS=[
    "mlp",
    "xgboost",
    "randomforest"
]

OUTLIERS=[
    None,
    "ZScore",
    "IsolationForest",
    "IQR"
]

RUNS=5


########################################
# LOGGING UTILITIES
########################################

def log_text(message):

    timestamp=time.strftime("%Y-%m-%d %H:%M:%S")

    line=f"[{timestamp}] {message}"

    print(line)

    with open("results/experiment_log.txt","a") as f:

        f.write(line+"\n")


def experiment_done(dataset,model,outlier):

    path="results/summary.csv"

    if not os.path.exists(path):

        return False

    df=pd.read_csv(path)

    return ((df.dataset==dataset)&
            (df.model==model)&
            (df.outlier.astype(str)==str(outlier))).any()


########################################
# DATA LOADING
########################################

def load_json(path):

    return json.loads(Path(path).read_text())


def load_dataset(dataset):

    base=Path(DATA_PATH)/dataset

    def load_block(name):

        return {

            x:np.load(
                base/f"{name}_{x}.npy",
                allow_pickle=True
            )

            for x in ["train","val","test"]
        }

    N=load_block("N") if (base/"N_train.npy").exists() else None

    C=load_block("C") if (base/"C_train.npy").exists() else None

    y=load_block("y")

    info=load_json(base/"info.json")

    return N,C,y,info


########################################
# PREPROCESSING
########################################

def preprocess_train(X):

    X=pd.DataFrame(X)

    numeric_cols=X.select_dtypes(include=[np.number]).columns

    means={}
    stds={}

    for col in numeric_cols:

        mean=X[col].mean()

        X[col]=X[col].fillna(mean)

        std=X[col].std()

        if std==0:

            std=1

        X[col]=(X[col]-mean)/std

        means[col]=mean
        stds[col]=std

    X = X.fillna(0).infer_objects(copy=False)

    return X.values,means,stds


def preprocess_test(X,means,stds):

    X=pd.DataFrame(X)

    for col in means:

        if col in X:

            X[col]=X[col].fillna(means[col])

            X[col]=(X[col]-means[col])/stds[col]

    X = X.fillna(0).infer_objects(copy=False)

    return X.values


########################################
# OUTLIERS
########################################

def detect_outliers(X,method,seed):

    # ensure numeric numpy array
    X = np.asarray(X)

    # convert possible object dtype safely
    if X.dtype == object:

        X = pd.DataFrame(X).apply(
            pd.to_numeric,
            errors="coerce"
        ).values

    # replace NaN produced by coercion
    X = np.nan_to_num(X)

    # ensure stable numeric type
    X = X.astype(np.float64)

    CONTAMINATION = 0.05

    if method is None:

        return np.ones(len(X),dtype=bool)

    if method=="ZScore":

        mean = X.mean(0)

        std = X.std(0)

        std[std == 0] = 1

        z = np.abs((X-mean)/std)

        return (z < 3).all(1)

    if method=="IQR":

        Q1 = np.percentile(X,25,axis=0)

        Q3 = np.percentile(X,75,axis=0)

        IQR = Q3-Q1

        lower = Q1-1.5*IQR

        upper = Q3+1.5*IQR

        return ((X>=lower)&
                (X<=upper)).all(1)

    if method=="IsolationForest":

        clf = IsolationForest(

            n_estimators=200,

            contamination=CONTAMINATION,

            random_state=seed
        )

        return clf.fit_predict(X)==1

    if method=="OneClassSVM":

        clf = OneClassSVM(

            nu=CONTAMINATION,

            kernel="rbf",

            gamma="scale"
        )

        return clf.fit_predict(X)==1

    raise ValueError("Unknown method")


########################################
# MODELS
########################################

def make_model(name,task,seed):

    name=name.lower()

    if name=="mlp":

        if task=="classification":

            return MLPClassifier(

                hidden_layer_sizes=(100,),

                max_iter=500,

                random_state=seed,

                shuffle=True
            )

        return MLPRegressor(

            hidden_layer_sizes=(100,),

            max_iter=500,

            random_state=seed
        )

    if name=="xgboost":

        if task=="classification":

            return XGBClassifier(

                use_label_encoder=False,

                eval_metric="logloss",

                random_state=seed
            )

        return XGBRegressor(random_state=seed)

    if name=="randomforest":

        if task=="classification":

            return RandomForestClassifier(

                n_estimators=100,

                random_state=seed
            )

        return RandomForestRegressor(

            n_estimators=100,

            random_state=seed
        )

    if name=="linear":

        if task=="classification":

            return LogisticRegression(max_iter=1000)

        return LinearRegression()

    raise ValueError("Unknown model")


########################################
# SINGLE RUN
########################################

def single_run(dataset,model,outlier,seed):

    np.random.seed(seed)

    N,C,y,info=load_dataset(dataset)

    # Ensure N only contains numeric data
    
    if N is not None:
    
        N_df = pd.DataFrame(N["train"])
    
        numeric_cols = N_df.select_dtypes(include=[np.number]).columns
    
        if len(numeric_cols) != N_df.shape[1]:
    
            # split numeric and categorical automatically
            numeric_data = N_df[numeric_cols].values
    
            categorical_data = N_df.drop(columns=numeric_cols).values
    
            N["train"] = numeric_data
    
            if C is None:
    
                C = {}
    
                C["train"] = categorical_data
    
                C["test"] = pd.DataFrame(N["test"]).drop(columns=numeric_cols).values
    
            else:
    
                C["train"] = np.hstack([C["train"],categorical_data])

    y_train=y["train"].ravel()

    y_test=y["test"].ravel()

    perm=np.random.permutation(len(y_train))

    y_train=y_train[perm]

    N["train"]=N["train"][perm]

    if C is not None:

        C["train"]=C["train"][perm]

    if N is not None:

        before=len(N["train"])
    
    else:
    
        before=len(C["train"])

    mask=detect_outliers(

        N["train"],

        outlier,

        seed
    )

    N["train"]=N["train"][mask]

    y_train=y_train[mask]

    if C is not None:

        C["train"]=C["train"][mask]

    after=len(N["train"])

    removed=before-after

    if C is not None:

        X_train=np.hstack([

            N["train"],

            C["train"]

        ])

        X_test=np.hstack([

            N["test"],

            C["test"]

        ])

    else:

        X_train=N["train"]

        X_test=N["test"]

    # convert to DataFrame for safe handling
    X_train = pd.DataFrame(X_train)
    X_test = pd.DataFrame(X_test)
    
    # encode categorical columns
    for col in X_train.columns:
    
        if X_train[col].dtype == object:
    
            le = LabelEncoder()
    
            X_train[col] = le.fit_transform(X_train[col].astype(str))
    
            X_test[col] = X_test[col].astype(str)
    
            # handle unseen values safely
            X_test[col] = X_test[col].map(
                lambda x: le.transform([x])[0] if x in le.classes_ else 0
            )
    
    # now preprocess numeric scaling
    X_train,means,stds = preprocess_train(X_train)
    
    X_test = preprocess_test(
        X_test,
        means,
        stds
    )

    perm=np.random.permutation(len(X_train))

    X_train=X_train[perm]

    y_train=y_train[perm]

    task=info["task_type"]

    if task!="regression":

        task="classification"

    if task=="classification":

        le=LabelEncoder()

        y_train=le.fit_transform(y_train)

        y_test=le.transform(y_test)

    model=make_model(

        model,

        task,

        seed
    )

    start=time.time()

    model.fit(X_train,y_train)

    train_time=time.time()-start

    score=evaluate(

        model,

        X_test,

        y_test,

        task
    )

    return {

        "score":score,

        "train_before":before,

        "train_after":after,

        "removed":removed,

        "removed_pct":removed/before if before>0 else 0,

        "time":train_time
    }


########################################
# EVALUATION
########################################

def evaluate(model,X,y,task):

    pred=model.predict(X)

    if task=="classification":

        return accuracy_score(y,pred)

    return mean_squared_error(y,pred)


########################################
# EXPERIMENT LOOP
########################################

def run_experiment(dataset,model,outlier):

    scores=[]

    removed=[]

    removed_pct=[]

    times=[]

    train_sizes=[]

    for i in range(RUNS):

        seed=42+i

        result=single_run(

            dataset,

            model,

            outlier,

            seed
        )

        scores.append(result["score"])

        removed.append(result["removed"])

        removed_pct.append(result["removed_pct"])

        times.append(result["time"])

        train_sizes.append(result["train_before"])

        log_text(

        f"{dataset} {model} {outlier} "

        f"run={i} "

        f"score={result['score']:.5f} "

        f"removed={result['removed']} "

        f"({result['removed_pct']:.3f})"
        )

    scores=np.array(scores)

    return {

        "mean":scores.mean(),

        "std":scores.std(),

        "min":scores.min(),

        "max":scores.max(),

        "train_size":int(np.mean(train_sizes)),

        "removed_mean":np.mean(removed),

        "removed_pct":np.mean(removed_pct),

        "time_mean":np.mean(times),

        "all":scores.tolist()
    }


########################################
# LOGGING
########################################

def init_logs():

    os.makedirs("results",exist_ok=True)

    if not os.path.exists("results/summary.csv"):

        with open("results/summary.csv","w") as f:

            f.write(

            "dataset,model,outlier,mean,std,min,max,train_size,removed,removed_pct,time\n"

            )

    if not os.path.exists("results/runs.csv"):

        with open("results/runs.csv","w") as f:

            f.write(

            "dataset,model,outlier,run,seed,score\n"

            )

    with open("results/experiment_log.txt","a") as f:

        f.write("\n====================\n")

        f.write(

        "START "+time.strftime("%Y-%m-%d %H:%M:%S")+"\n"
        )


def log_summary(dataset,model,outlier,res):

    with open("results/summary.csv","a") as f:

        f.write(

        f"{dataset},{model},{outlier},"

        f"{res['mean']:.5f},"

        f"{res['std']:.5f},"

        f"{res['min']:.5f},"

        f"{res['max']:.5f},"

        f"{res['train_size']},"

        f"{res['removed_mean']:.2f},"

        f"{res['removed_pct']:.4f},"

        f"{res['time_mean']:.3f}\n"
        )


def log_runs(dataset,model,outlier,res):

    with open("results/runs.csv","a") as f:

        for i,score in enumerate(res["all"]):

            seed=42+i

            f.write(

            f"{dataset},{model},{outlier},{i},{seed},{score:.5f}\n"

            )


########################################
# MAIN
########################################

def main():

    init_logs()

    log_text("Experiment started")

    total = len(DATASETS)*len(MODELS)*len(OUTLIERS)

    total_runs = total * RUNS

    log_text("===== EXPERIMENT SETUP =====")

    log_text(f"Datasets: {DATASETS}")

    log_text(f"Models: {MODELS}")

    log_text(f"Outliers: {OUTLIERS}")

    log_text(f"Runs per experiment: {RUNS}")

    log_text(f"Total experiments: {total}")

    log_text(f"Total model trainings: {total_runs}")

    log_text("============================")

    counter = 0

    for dataset in DATASETS:

        log_text(f"==== DATASET {dataset} ====")

        try:

            # quick dataset test
            load_dataset(dataset)

        except Exception as e:

            log_text(f"DATASET FAILED {dataset}")

            log_text(str(e))

            continue

        for model in MODELS:

            for outlier in OUTLIERS:

                counter += 1

                progress = 100*counter/total

                label = f"{dataset} {model} {outlier}"

                if experiment_done(dataset,model,outlier):

                    log_text(
                    f"SKIP {counter}/{total} "
                    f"({progress:.1f}%) "
                    f"{label}"
                    )

                    continue

                try:

                    log_text(
                    f"RUN {counter}/{total} "
                    f"({progress:.1f}%) "
                    f"{label}"
                    )

                    res = run_experiment(

                        dataset,
                        model,
                        outlier
                    )

                    log_text(

                    f"RESULT {label} "

                    f"mean={res['mean']:.5f} "

                    f"std={res['std']:.5f} "

                    f"removed={res['removed_mean']:.1f} "

                    f"({res['removed_pct']:.3f})"
                    )

                    log_summary(

                        dataset,
                        model,
                        outlier,
                        res
                    )

                    log_runs(

                        dataset,
                        model,
                        outlier,
                        res
                    )

                except Exception as e:

                    log_text(f"EXPERIMENT FAILED {label}")

                    log_text(str(e))

                    continue

        log_text(f"==== DATASET {dataset} DONE ====")

    log_text("Experiment finished")

if __name__=="__main__":

    main()

[2026-03-19 11:50:47] Experiment started
[2026-03-19 11:50:47] ===== EXPERIMENT SETUP =====
[2026-03-19 11:50:47] Datasets: ['Bank_Customer_Churn_Dataset', 'cmc', 'connect-4', 'credit-g', 'electricity', 'elevators', 'eucalyptus', 'eye_movements', 'house_16H', 'kc1', 'phoneme', 'pol', 'socmob', 'splice', 'vehicle']
[2026-03-19 11:50:47] Models: ['mlp', 'xgboost', 'randomforest']
[2026-03-19 11:50:47] Outliers: [None, 'ZScore', 'IsolationForest', 'IQR']
[2026-03-19 11:50:47] Runs per experiment: 5
[2026-03-19 11:50:47] Total experiments: 180
[2026-03-19 11:50:47] Total model trainings: 900
[2026-03-19 11:50:47] ============================
[2026-03-19 11:50:47] ==== DATASET Bank_Customer_Churn_Dataset ====
[2026-03-19 11:50:47] RUN 1/180 (0.6%) Bank_Customer_Churn_Dataset mlp None
[2026-03-19 11:50:53] Bank_Customer_Churn_Dataset mlp None run=0 score=0.84500 removed=0 (0.000)


/home/lisp/code/tabular-nns/env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


[2026-03-19 11:51:00] Bank_Customer_Churn_Dataset mlp None run=1 score=0.83300 removed=0 (0.000)
[2026-03-19 11:51:08] Bank_Customer_Churn_Dataset mlp None run=2 score=0.83850 removed=0 (0.000)


/home/lisp/code/tabular-nns/env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


[2026-03-19 11:51:14] Bank_Customer_Churn_Dataset mlp None run=3 score=0.83850 removed=0 (0.000)
[2026-03-19 11:51:21] Bank_Customer_Churn_Dataset mlp None run=4 score=0.84650 removed=0 (0.000)
[2026-03-19 11:51:21] RESULT Bank_Customer_Churn_Dataset mlp None mean=0.84030 std=0.00491 removed=0.0 (0.000)
[2026-03-19 11:51:21] RUN 2/180 (1.1%) Bank_Customer_Churn_Dataset mlp ZScore
[2026-03-19 11:51:27] Bank_Customer_Churn_Dataset mlp ZScore run=0 score=0.83050 removed=136 (0.021)
[2026-03-19 11:51:33] Bank_Customer_Churn_Dataset mlp ZScore run=1 score=0.82650 removed=136 (0.021)


...

[2026-03-19 12:20:22] vehicle mlp None run=4 score=0.84706 removed=0 (0.000)
[2026-03-19 12:20:22] RESULT vehicle mlp None mean=0.83647 std=0.01141 removed=0.0 (0.000)
[2026-03-19 12:20:22] SKIP 170/180 (94.4%) vehicle mlp ZScore
[2026-03-19 12:20:22] SKIP 171/180 (95.0%) vehicle mlp IsolationForest
[2026-03-19 12:20:22] SKIP 172/180 (95.6%) vehicle mlp IQR
[2026-03-19 12:20:22] RUN 173/180 (96.1%) vehicle xgboost None
[2026-03-19 12:20:22] vehicle xgboost None run=0 score=0.77647 removed=0 (0.000)
[2026-03-19 12:20:22] vehicle xgboost None run=1 score=0.77647 removed=0 (0.000)
[2026-03-19 12:20:22] vehicle xgboost None run=2 score=0.77647 removed=0 (0.000)


/home/lisp/code/tabular-nns/env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [12:20:22] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/lisp/code/tabular-nns/env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [12:20:22] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[2026-03-19 12:20:22] vehicle xgboost None run=3 score=0.77647 removed=0 (0.000)
[2026-03-19 12:20:23] vehicle xgboost None run=4 score=0.77647 removed=0 (0.000)
[2026-03-19 12:20:23] RESULT vehicle xgboost None mean=0.77647 std=0.00000 removed=0.0 (0.000)
[2026-03-19 12:20:23] SKIP 174/180 (96.7%) vehicle xgboost ZScore
[2026-03-19 12:20:23] SKIP 175/180 (97.2%) vehicle xgboost IsolationForest
[2026-03-19 12:20:23] SKIP 176/180 (97.8%) vehicle xgboost IQR
[2026-03-19 12:20:23] RUN 177/180 (98.3%) vehicle randomforest None


/home/lisp/code/tabular-nns/env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [12:20:23] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[2026-03-19 12:20:23] vehicle randomforest None run=0 score=0.72941 removed=0 (0.000)
[2026-03-19 12:20:23] vehicle randomforest None run=1 score=0.71765 removed=0 (0.000)
[2026-03-19 12:20:23] vehicle randomforest None run=2 score=0.73529 removed=0 (0.000)
[2026-03-19 12:20:23] vehicle randomforest None run=3 score=0.75294 removed=0 (0.000)
[2026-03-19 12:20:23] vehicle randomforest None run=4 score=0.72941 removed=0 (0.000)
[2026-03-19 12:20:23] RESULT vehicle randomforest None mean=0.73294 std=0.01153 removed=0.0 (0.000)
[2026-03-19 12:20:23] SKIP 178/180 (98.9%) vehicle randomforest ZScore
[2026-03-19 12:20:23] SKIP 179/180 (99.4%) vehicle randomforest IsolationForest
[2026-03-19 12:20:23] SKIP 180/180 (100.0%) vehicle randomforest IQR
[2026-03-19 12:20:23] ==== DATASET vehicle DONE ====
[2026-03-19 12:20:23] Experiment finished


In [6]:
res = run_experiment(
    dataset="electricity",
    model="xgboost",
    outlier=None
)

print("\nRESULT:")
print("Mean:", res["mean"])
print("Std:", res["std"])
print("All runs:", res["all"])

/home/lisp/code/tabular-nns/env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [23:27:51] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/lisp/code/tabular-nns/env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [23:27:52] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/lisp/code/tabular-nns/env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [23:27:52] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/lisp/code/tabular-nns/env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [23:27:52] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/lisp/code/


RESULT:
Mean: 0.7837360697340836
Std: 0.0
All runs: [0.7837360697340836, 0.7837360697340836, 0.7837360697340836, 0.7837360697340836, 0.7837360697340836]


In [13]:
run_experiment("cmc","mlp",None)

{'mean': np.float64(0.5850847457627119),
 'std': np.float64(0.007906375450637698),
 'min': np.float64(0.576271186440678),
 'max': np.float64(0.5966101694915255),
 'removed_mean': np.float64(0.0),
 'time_mean': np.float64(0.26859517097473146),
 'all': [0.576271186440678,
  0.5864406779661017,
  0.5966101694915255,
  0.5898305084745763,
  0.576271186440678]}